# Parsing one page: Docling, PaddleOCR-VL and Qwen VL

Notebook demo of the AI2Future workshop *Can We Rely on Extracted Documents? Parsing Methods Compared*
(Filip Pavičić, Atomic Intelligence).

We follow one page of a synthetic lab report through a parsing pipeline: the regions a parser finds, the
Markdown it writes, the errors it makes and the checks that catch some of them. Docling runs here, on the
CPU. PaddleOCR-VL and Qwen VL need a GPU, so their outputs on the same page are loaded from `data/cached/`.

Run the cells from top to bottom. Sections 5 and 7 can be skipped.

## Setup

On Colab, the first cell clones the repository and installs the dependencies, which takes a few minutes.
Locally, install as described in the README and start Jupyter from the repository. The second cell
loads Docling's models, and downloads them the first time.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and Path.cwd().name != "document-understanding-demo":
    if not Path("document-understanding-demo").exists():
        !git clone --quiet --depth 1 https://github.com/Atomic-Intelligence/document-understanding-demo.git
    %cd document-understanding-demo
    # pip warns that google-colab pins an older requests than Docling needs; Colab works with the
    # newer one, so the output is shown only if the install fails.
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--progress-bar", "off", "-e", "."],
                         capture_output=True, text=True)
    if pip.returncode != 0:
        print(pip.stdout, pip.stderr)
        raise RuntimeError("Installing the dependencies failed; pip's output is above.")
    print("Dependencies installed.")

# The repository root, whether Jupyter was started in notebooks/ or in the root
ROOT = Path.cwd() if (Path.cwd() / "src" / "dudemo").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from dudemo import checks, data, docling_run, show, stitch

converter = docling_run.make_converter()
print("Docling is ready.")

## 1. The page

A synthetic Croatian lab report from our page generator: patient fields, two tables of results with
units and reference ranges, and a flag column (H for high, L for low). The page is an image, so there is
no text layer to copy: every parser has to read the pixels.

Left: the regions of the page, coloured by type. Right: the Markdown a perfect parser would write.

In [ ]:
PAGE = "lab_report_1"
truth = data.ground_truth(PAGE)
show.parse(truth)

## 2. Parse it with Docling

Docling finds the regions with a layout model, reads the text with OCR and rebuilds each table with a
table-structure model. No part of it generates text: it can misread a character, but it cannot invent
a sentence.

In [ ]:
docling = docling_run.run(converter, data.page_path(PAGE))
print(f"Docling read the page in {docling.seconds:.0f} seconds on this machine.")
show.parse(docling)

**Question for the room:** what did Docling get wrong in the two tables?

## 3. The same page, three other parsers

- **PaddleOCR-VL** finds the regions first, then a small vision-language model reads each region.
- **Qwen VL, general** reads the whole page in one pass and writes the regions and their text together.
  It is a general model, not made for documents.
- **Qwen VL, fine-tuned** is the same model after our fine-tuning on generated pages.

Their outputs were made on a GPU and are loaded here.

In [ ]:
for model in data.CACHED_MODELS:
    show.parse(data.cached(PAGE, model))

**Question for the room:** where did the general model put its regions?

## 4. Three kinds of error

Each example shows the row on the page, the row as the page has it, and the row as the parser wrote it.
Cells that differ are marked red.

- **Visibly broken**: anyone reading the output sees it.
- **Structure wrong**: the text may be right, but the shape is not.
- **Passes as correct**: a wrong value that looks right.

In [ ]:
outputs = {"docling": docling, **{model: data.cached(PAGE, model) for model in data.CACHED_MODELS}}
for example in data.examples():
    show.error(example, outputs[example["model"]], truth)

## 5. General and fine-tuned Qwen VL

The same model, prompt and settings; only the weights differ. On a second report, every region of the
page that the model missed is outlined in dashed red.

*Can be skipped.*

In [ ]:
PAGE_2 = "lab_report_2"
show.side_by_side(data.cached(PAGE_2, "qwen_base"), data.cached(PAGE_2, "qwen_finetuned"),
                  missed_from=data.ground_truth(PAGE_2))

Fine-tuning taught the model the format: regions in the right place, tables as tables. It did not make
every value right: on this page the fine-tuned model writes the doctor *Dr. Ivić* as *Dr. Tić*, a
well-formed name that nothing in the output marks as wrong.

## 6. Checks after extraction

Three rules read only Docling's output, with no ground truth: is the unit a known unit, does the range
look like a range, and does the H or L flag fit the result and the range. A ✗ sends the row to a
person; – means there was nothing to check.

In [ ]:
show.table(checks.check_results(docling))

A rule cannot tell whether a name or a record number is right. For those, the only check is a second
parser that reads the page differently: where the two disagree, a person looks. Here the general
Qwen VL model is compared with PaddleOCR-VL on the patient fields.

In [ ]:
show.table(checks.compare_fields(outputs["qwen_base"], outputs["paddleocr_vl"]))

## 7. A table across pages

Parsers read one page at a time. Here the report is cut through its first table, as if the table went on
to a second page. The rows at the top of page 2 have no header, so the parser does not know what the
columns are.

*Can be skipped.*

In [ ]:
end_of_page_1, top_of_page_2 = stitch.split_table(PAGE, after_row="Hematokrit (Hct)")
show.page_break(end_of_page_1, top_of_page_2)
show.markdown(docling_run.run(converter, stitch.with_margin(top_of_page_2)), "Docling on the top of page 2")

The repair: crop both parts, stack them into one image and run the same parser again, so that the header
travels with the rows. Nothing is retrained.

In [ ]:
both_parts = stitch.stack(end_of_page_1, top_of_page_2)
show.image(both_parts, "Both parts stacked")
show.markdown(docling_run.run(converter, stitch.with_margin(both_parts)), "Docling on both parts stacked")

## 8. Try your own document

Set `MY_FILE` to the path of a PDF or an image and run the cell. On Colab, upload the file with the file
browser on the left; it lands in `/content/`, so the path is `/content/<file name>`.

- Docling: https://github.com/docling-project/docling
- PaddleOCR-VL: https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6
- Qwen2.5-VL: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct

In [ ]:
MY_FILE = "path/to/your/document.pdf"

if Path(MY_FILE).exists():
    show.parse(docling_run.run(converter, MY_FILE))
else:
    print("Set MY_FILE to the path of a PDF or an image, then run this cell again.")